# Topic 2 - Introducing LLMs

## What lies inside ChatGPT?

By the end of this topic you will be able to:
- Explain the three transformer architecture families (encoder, decoder, encoder-decoder) and
  choose the right one for a given task
- Tokenize a real Barclays complaint text, inspect token IDs, and decode them back
- Get dense embeddings from a pretrained model and measure semantic similarity between complaints
- Run inference with a small pretrained model (distilbert-base-uncased, distilgpt2) directly
  in the Studio notebook kernel
- Describe the Generative AI project lifecycle and how it differs from traditional ML

## Context: Our Barclays Complaint System

In Topic 1 we decided we want an LLM to handle customer complaints intelligently.
Now we need to answer: which kind of LLM? And what does "LLM" actually mean in code?

We will work with real-looking Barclays complaint text throughout this topic.
No training in this topic -- pure inference and exploration.

In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup -- run once at the start of every notebook.
# Do NOT skip this cell. Pinned versions are required for SageMaker Studio compatibility.

!pip install -q \
    "transformers>=4.53,<4.54" \
    "accelerate>=1.0.0" \
    "tokenizers>=0.21,<0.22" \
    "datasets>=2.18.0,<3.0.0" \
    "numpy<2" \
    "scikit-learn>=1.3.0,<2.0.0" \
    "sagemaker>=2.200.0,<3.0.0"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF_IMPORT"] = "1"  # prevent broken tf_keras import on this image

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline,
    DistilBertTokenizer,
    DistilBertModel,
    GPT2Tokenizer,
    GPT2LMHeadModel,
)
from sklearn.metrics.pairwise import cosine_similarity

print("torch version:", torch.__version__)
print("numpy version:", np.__version__)
print("Setup complete.")

In [ ]:
# Handoff from Topic 1: load the Barclays triage artifacts you produced there.
# In Topic 1 you wrote a triage system prompt and 5 test complaints, and saved them
# to S3. We load them now and extend the system: this topic tokenizes those same
# complaints and turns them into embeddings.
# The OpenAI client is NOT loaded - it is a live connection. Topic 2 does no API
# calls, so no client is needed here.

# Topics 1-3 have no sagemaker.Session(); resolve the course bucket directly.
import sagemaker
bucket = sagemaker.Session().default_bucket()

# --- S3 handoff helpers (course-wide, identical in every notebook) ---
import json, boto3, botocore

COURSE_PREFIX = "barclays-course"

def _handoff_key(topic_n, artifact):
    return f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"

def handoff_read(bucket, topic_n, artifact):
    """Read a JSON object from a topic's handoff prefix. Returns None if absent."""
    key = _handoff_key(topic_n, artifact)
    try:
        body = boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()
        print(f"Handoff loaded: s3://{bucket}/{key}")
        return json.loads(body)
    except botocore.exceptions.ClientError:
        print(f"No handoff found at s3://{bucket}/{key} (starting mid-course is fine).")
        return None

_t1 = handoff_read(bucket, 1, "triage_config.json")

if _t1 is not None:
    test_complaints = _t1["test_complaints"]
    triage_system_prompt = _t1["system_prompt"]
    print(f"Loaded {len(test_complaints)} test complaints from Topic 1.")
else:
    # Fallback: student is starting at Topic 2. Define a minimal local version
    # so the rest of the notebook runs. Same spirit as a lab safety-net cell.
    print("Using a local fallback complaint set so Topic 2 runs standalone.")
    test_complaints = [
        "I sent 1200 pounds to my sister three days ago and it still has not arrived.",
        "I just got an alert for a 350 pound transaction in Manchester I did not make.",
        "I cannot log in to the app; it texts a code to my old phone number.",
        "You charged me a 25 pound overdraft fee for being four hours overdrawn.",
        "300 pounds left my account to someone I have never heard of.",
    ]
    triage_system_prompt = "You are a complaint triage agent for Barclays Bank."

print("These are the complaints you triaged in Topic 1. Now we look inside the model:")
print("we tokenize them and turn them into embeddings.")


## Section 1 -- What Is a Token?

Before any LLM sees text, it converts it to numbers.
That conversion is called **tokenization**.

The Barclays complaint system receives thousands of messages like:

> "I've been charged twice for the same transaction on my Barclaycard. This is unacceptable
>  and I need a refund immediately."

To an LLM, that sentence is just a sequence of integers. Let's see how.

### Beat 1 -- The Naive Approach (and Why It Fails)

Suppose we just split on spaces and look up words in a dictionary.
What happens with unknown words, contractions, or rare financial terms?

In [ ]:
# Beat 1 -- Naive tokenization: split on spaces, assign sequential IDs.
# This is the WRONG approach. Run it and see what breaks.

complaint = (
    "I've been charged twice for the same transaction on my Barclaycard. "
    "This is unacceptable and I need a refund immediately."
)

# Build a tiny vocabulary from the complaint itself
words = complaint.split()
vocab = {word: idx for idx, word in enumerate(sorted(set(words)))}
print("Naive vocabulary size:", len(vocab))
print("Vocabulary:", vocab)

# Tokenize
naive_tokens = [vocab.get(word, -1) for word in words]
print("\nNaive token IDs:", naive_tokens)

# Problem 1: a new complaint uses words not in our tiny vocab
new_complaint = "My Barclaycard was fraudulently used at an ATM in Manchester."
new_tokens = [vocab.get(word, -1) for word in new_complaint.split()]
print("\nNew complaint token IDs:", new_tokens)
print("Number of UNKNOWN tokens (-1):", new_tokens.count(-1), "out of", len(new_tokens))

# Problem 2: contractions are not split -- "I've" is treated as one unit
print("\nDoes vocab contain 'I'?", "I" in vocab)
print("Does vocab contain \"I've\"?", "I've" in vocab)
print("Real models split I've into: ['I', \"'\", 've'] or ['I', \"'ve']")

### Beat 2 -- How Real LLMs Handle Tokenization

Real transformer models use **subword tokenization** (Byte-Pair Encoding for GPT-style,
WordPiece for BERT-style). Instead of a dictionary of whole words, they have a fixed
vocabulary of ~30,000 common subwords. Unknown words are split into known pieces.

"unacceptable" -> ["un", "##accept", "##able"] (BERT WordPiece)
"Barclaycard"  -> ["Bar", "clay", "card"] (GPT BPE)

This means the vocabulary is fixed and finite, but the model can handle ANY text --
including new financial product names -- by splitting them into known subpieces.

<!-- DIAGRAM: transformer-families -->

```mermaid
graph TD
    text["Raw text input"] --> tok["Subword tokenizer\n(BPE or WordPiece)"]
    tok --> ids["Token IDs\n(fixed vocab ~30k)"]
    ids --> enc["Encoder-only\n(BERT, DistilBERT)\nReads full sequence\nUse: classification, embeddings"]
    ids --> dec["Decoder-only\n(GPT-4o, Llama)\nGenerates left-to-right\nUse: text generation, chat"]
    ids --> encdec["Encoder-Decoder\n(Flan-T5, BART)\nEncodes input, decodes output\nUse: translation, summarization"]

    style tok fill:#ffd,stroke:#aa3
    style enc fill:#ddf,stroke:#33c
    style dec fill:#fda,stroke:#c63
    style encdec fill:#dfd,stroke:#3a3
```

*Figure: The three transformer families share the same subword tokenization front-end and diverge after token IDs are produced. Choice of family determines which tasks the model is suited for.*

The three transformer families all share this subword tokenization front-end.
They differ in what happens AFTER the tokens become numbers.

In [ ]:
# Beat 3 -- Real tokenization with DistilBERT (WordPiece, BERT-style).
# DistilBERT uses a ~30,000 word vocabulary of subwords.
# It is a smaller (66M param) distilled version of BERT -- perfect for demos.

print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
print(f"Vocabulary size: {tokenizer.vocab_size:,} subwords")

# Use the first complaint carried over from Topic 1 (loaded in the handoff cell above).
complaint = test_complaints[0]

# Step 1: Tokenize -- converts text to a dict with input_ids and attention_mask
encoded = tokenizer(complaint, return_tensors="pt")
print("\n--- Encoded output ---")
print("Keys:", list(encoded.keys()))
print("input_ids shape:", encoded["input_ids"].shape)  # [1, num_tokens]
print("attention_mask shape:", encoded["attention_mask"].shape)

# Step 2: Inspect the token IDs
token_ids = encoded["input_ids"][0].tolist()
print("\nToken IDs:", token_ids)
print("Number of tokens:", len(token_ids))

# Step 3: Convert IDs back to human-readable subwords
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("\nSubwords:", tokens)
# Note: [CLS] at start, [SEP] at end -- special BERT bookend tokens
# ## prefix means "continuation of previous subword"

# Step 4: Decode back to original text
decoded = tokenizer.decode(token_ids, skip_special_tokens=True)
print("\nDecoded back:", decoded)
print("Round-trip identical:", decoded.strip().lower() == complaint.strip().lower())

# Step 5: Show how an unknown financial term is split
rare_term = "Barclaycard fraudulently overdraft"
rare_encoded = tokenizer(rare_term)
rare_tokens = tokenizer.convert_ids_to_tokens(rare_encoded["input_ids"])
print("\nRare financial terms tokenized:", rare_tokens)

## Lab 1 -- Tokenization Explorer (Tier 1 Guided, ~15 min)

**Situation**: The Barclays complaint triage team wants to understand why certain messages
are flagged as "too long" by the LLM system. The system has a 512-token limit.
They need a simple function that counts tokens and warns when a complaint is near the limit.

**Task**: Build a complaint tokenizer that:
1. Tokenizes a complaint using distilbert-base-uncased
2. Returns the list of subword tokens
3. Returns the token count
4. Prints a warning if count is above 400 (leaves headroom before the 512 limit)

**Action**: Fill in the YOUR CODE sections below.

**Result**: Running the verification cell will show token counts for three test complaints
and flag the long one correctly.

### Steps
1. Call `tokenizer(complaint, return_tensors="pt")` to encode
2. Use `.convert_ids_to_tokens(...)` to get human-readable subwords
3. Count how many tokens were produced
4. Compare count to the 400-token threshold

In [ ]:
# Lab 1 -- Tokenization Explorer (SOLUTION)
# The tokenizer is already loaded above as `tokenizer`.

def analyze_complaint_tokens(complaint_text, threshold=400):
    """
    Tokenize a complaint and return analysis results.

    Parameters
    ----------
    complaint_text : str
        Raw complaint text from a Barclays customer.
    threshold : int
        Token count above which to print a warning.

    Returns
    -------
    dict with keys: tokens (list[str]), count (int), over_limit (bool)
    """
    # Step 1: Encode using the DistilBERT tokenizer
    # return_tensors="pt" gives us PyTorch tensors for the input_ids
    encoded = tokenizer(complaint_text, return_tensors="pt")

    # Step 2: Extract token IDs as a plain Python list
    # encoded["input_ids"] shape is [1, num_tokens]; index [0] drops the batch dim
    token_ids = encoded["input_ids"][0].tolist()

    # Step 3: Convert integer IDs back to string subwords
    # This is useful for inspecting how the model "sees" the complaint
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    # Step 4: Count all tokens including [CLS] and [SEP] special tokens
    count = len(tokens)

    # Step 5: Check whether we are over the safe threshold
    # Keeping headroom below 512 avoids silent truncation in downstream pipelines
    over_limit = count > threshold

    if over_limit:
        print(f"WARNING: complaint has {count} tokens (limit is {threshold}). Consider truncating.")

    return {"tokens": tokens, "count": count, "over_limit": over_limit}


# Test complaints
# Note: the test_complaints loaded from Topic 1 are also valid inputs to
# analyze_complaint_tokens; the strings below exercise specific length thresholds.
short_complaint = "My card was declined at a supermarket."
medium_complaint = (
    "I have been a loyal Barclays customer for 15 years. Last Tuesday I attempted "
    "to make a payment of 2,500 pounds to my solicitor for a property purchase. "
    "The payment was blocked without any notification and I am now facing penalties. "
    "I need this resolved urgently or I will escalate to the Financial Ombudsman."
)
long_complaint = " ".join(["I am extremely dissatisfied with the service."] * 25)

# Call the function on all three complaints
result_short  = analyze_complaint_tokens(short_complaint)
result_medium = analyze_complaint_tokens(medium_complaint)
result_long   = analyze_complaint_tokens(long_complaint)

In [ ]:
# Lab 1 verification -- run after completing the lab
assert result_short is not None, "result_short is still None -- did you call analyze_complaint_tokens?"
assert result_medium is not None, "result_medium is still None"
assert result_long is not None, "result_long is still None"

assert isinstance(result_short["tokens"], list), "tokens must be a list of strings"
assert result_short["count"] > 0, "count must be positive"
assert result_short["over_limit"] == False, "short complaint should not be over limit"
assert result_long["over_limit"] == True, "long complaint (25 repetitions) should be over limit"

print("Lab 1 passed!")
print(f"  short complaint:  {result_short['count']} tokens, over_limit={result_short['over_limit']}")
print(f"  medium complaint: {result_medium['count']} tokens, over_limit={result_medium['over_limit']}")
print(f"  long complaint:   {result_long['count']} tokens, over_limit={result_long['over_limit']}")

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if result_short is None or result_medium is None or result_long is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")

    def analyze_complaint_tokens(complaint_text, threshold=400):
        enc = tokenizer(complaint_text, return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        tokens = tokenizer.convert_ids_to_tokens(ids)
        count = len(tokens)
        over_limit = count > threshold
        if over_limit:
            print(f"WARNING: {count} tokens (limit {threshold}).")
        return {"tokens": tokens, "count": count, "over_limit": over_limit}

    result_short  = analyze_complaint_tokens(short_complaint)
    result_medium = analyze_complaint_tokens(medium_complaint)
    result_long   = analyze_complaint_tokens(long_complaint)
    print("Safety-net complete.")

### Lab 1 Stretch (for fast finishers)

Compare tokenization between DistilBERT (WordPiece) and DistilGPT2 (BPE) on the same complaint.
- Load `GPT2Tokenizer.from_pretrained("distilgpt2")`
- Tokenize the medium_complaint with both tokenizers
- Compare: which produces more tokens? Which subwords look different?
- Why does GPT2 tokenizer NOT add [CLS] and [SEP]?

### Homework Extension

Build a token budget checker for a batch of complaints loaded from a list.
Given 20 complaints, produce a summary table with columns:
  complaint_id | token_count | over_limit | first_10_tokens
Use `pandas.DataFrame` for the table.
Identify: what is the longest complaint in the batch? What is the average token count?
This is the kind of preprocessing check the Barclays data team would run before fine-tuning.

## Peer Discussion (3 min)

With the person next to you:

1. A customer pastes their complaint in Welsh. The model was trained mostly on English.
   What do you predict happens to the token count? What might happen to the output quality?

2. Barclays wants to process 10,000 complaints per hour through the LLM.
   Each complaint costs money proportional to its token count (input + output tokens).
   How would you use what you just built to estimate and control that cost?

3. Why does it matter that we can DECODE token IDs back to text?
   Think about debugging and compliance audit trails.

## Section 2 -- From Tokens to Meaning: Embeddings

Token IDs are just integers -- 2345 does not "mean" anything on its own.
The transformer converts those integers into **dense vectors** (embeddings) that
encode semantic meaning.

The magic: two complaint sentences that mean the same thing will have
**similar embedding vectors**, even if they share no words.

"I was charged twice." and "A duplicate payment was taken." should be close together.
"I love the Barclays app." should be far away from both.

### Beat 1 -- Naive Text Comparison Without Embeddings (Broken)

Suppose we try to find "similar complaints" using string matching.

In [ ]:
# Beat 1 -- Naive similarity: exact string match and word-overlap (Jaccard).
# This is WRONG for semantic similarity. Run it and see the failures.

def jaccard_similarity(text_a, text_b):
    """Word overlap between two texts. Range [0, 1]."""
    set_a = set(text_a.lower().split())
    set_b = set(text_b.lower().split())
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union) if union else 0.0

# Three complaint pairs
pairs = [
    # Should be HIGH similarity (same meaning, different words)
    ("I was charged twice for the same payment.",
     "A duplicate transaction appeared on my account."),
    # Should be LOW similarity (different topics)
    ("I was charged twice for the same payment.",
     "My mortgage application has been delayed by three weeks."),
    # Should be MEDIUM similarity (same topic, very different wording)
    ("I've been a victim of card fraud.",
     "Someone made unauthorized purchases with my Barclaycard."),
]

print("Jaccard (word overlap) similarity scores:")
print("-" * 70)
for a, b in pairs:
    score = jaccard_similarity(a, b)
    print(f"Score: {score:.3f}")
    print(f"  A: {a}")
    print(f"  B: {b}")
    print()

# Expected: all three scores are LOW (around 0.05-0.15) because the words differ.
# Pair 1 SHOULD be high but is low -- Jaccard fails completely.
print("PROBLEM: Pair 1 should be highest (same meaning) but Jaccard scores it low.")
print("Jaccard only sees word overlap, not meaning.")

### Beat 2 -- How Embeddings Capture Meaning

A transformer encoder processes a full sentence and produces a single dense vector
(typically 768 numbers for BERT-base, 384 for DistilBERT-base).

Words that appear in similar contexts during pretraining end up in similar regions
of this 768-dimensional space. Two sentences about duplicate charges will be
"near" each other even with different words.

<!-- DIAGRAM: genai-lifecycle -->

```mermaid
graph TD
    problem["Define the problem\n(what does success look like?)"] --> data["Collect and clean data\n(complaints, transactions, labels)"]
    data --> embed["Embed text\n(encoder model -> dense vectors)"]
    embed --> search["Semantic search\n(find similar complaints)"]
    search --> choose{"Build or use\npretrained model?"}
    choose -->|"Use as-is"| prompt["Prompt engineering\n(system prompt + examples)"]
    choose -->|"Adapt"| finetune["Fine-tune\n(labeled data + training job)"]
    prompt --> eval["Evaluate\n(accuracy, latency, cost)"]
    finetune --> eval
    eval --> deploy["Deploy to production\n(endpoint or batch job)"]
    deploy --> monitor["Monitor and retrain\n(drift, new complaint types)"]

    style embed fill:#ddf,stroke:#33c
    style search fill:#ddf,stroke:#33c
    style deploy fill:#dfd,stroke:#3a3
```

*Figure: The GenAI project lifecycle -- embeddings and semantic search are standard production steps. The build-vs-use decision determines whether prompt engineering or fine-tuning is the next step.*

The diagram shows the GenAI project lifecycle -- we will come back to this in Section 4.
For now, notice that "embeddings" and "semantic search" are standard production steps
in the GenAI workflow, not research novelties.

In [ ]:
# Beat 3 -- Real sentence embeddings using DistilBERT encoder.
# DistilBERT is an encoder-only model: it reads the WHOLE sentence bidirectionally.
# We extract contextual embeddings and mean-pool them to get a sentence-level vector.

print("Loading DistilBERT model (66M parameters)...")
bert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")
bert_model.eval()  # inference mode -- no gradient tracking needed
print("Model loaded.")

def get_embedding(text):
    """
    Encode a sentence and return a 768-dim numpy embedding vector.
    Uses mean pooling across all token positions (more stable than CLS alone).
    """
    # Tokenize with padding/truncation
    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    with torch.no_grad():
        outputs = bert_model(**inputs)
    # outputs.last_hidden_state shape: [1, seq_len, 768]
    # Mean pool across the sequence dimension (ignore padding via attention_mask)
    attention_mask = inputs["attention_mask"]  # [1, seq_len]
    token_embeddings = outputs.last_hidden_state  # [1, seq_len, 768]
    # Expand mask to match embedding dimensions
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
    embedding = (sum_embeddings / sum_mask).squeeze().numpy()
    return embedding  # shape: (768,)


# Get embeddings for complaint pairs
sentences = [
    "I was charged twice for the same payment.",
    "A duplicate transaction appeared on my account.",
    "My mortgage application has been delayed by three weeks.",
    "Someone made unauthorized purchases with my Barclaycard.",
    "I've been a victim of card fraud.",
    "I love the new Barclays mobile app design.",
]

print("\nGenerating embeddings (this takes a few seconds)...")
embeddings = [get_embedding(s) for s in sentences]
embeddings_matrix = np.stack(embeddings)  # shape: (6, 768)
print(f"Embeddings shape: {embeddings_matrix.shape}")

# Compute pairwise cosine similarity
sim_matrix = cosine_similarity(embeddings_matrix)

print("\nCosine similarity matrix (higher = more similar):")
print("Sentences:")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")
print()
print("Similarity matrix (rows x cols = sentence indices):")
print(np.round(sim_matrix, 3))

# Highlight the interesting pairs
print("\nKey pairs:")
print(f"  [0] vs [1] (duplicate charge -- SHOULD be high): {sim_matrix[0,1]:.3f}")
print(f"  [0] vs [2] (mortgage -- SHOULD be low):          {sim_matrix[0,2]:.3f}")
print(f"  [3] vs [4] (card fraud -- SHOULD be high):       {sim_matrix[3,4]:.3f}")
print(f"  [0] vs [5] (mobile app -- SHOULD be low):        {sim_matrix[0,5]:.3f}")

## Lab 2 -- Complaint Similarity Matrix (Tier 1 Guided, ~15 min)

**Situation**: The Barclays complaint routing team has a dataset of 10 complaints
across different categories: card fraud, duplicate charges, mortgage delays,
and app feedback. They want to automatically identify which complaints are
"the same issue described differently" so they can route them to the same specialist.

**Task**: Build a similarity search function that:
1. Takes a list of complaint texts
2. Computes DistilBERT embeddings for all of them
3. Builds a full pairwise cosine similarity matrix
4. Finds the single most similar pair in the dataset (excluding self-similarity on the diagonal)
5. Finds the single least similar pair

**Action**: Fill in the YOUR CODE sections below. Follow the numbered steps.

**Result**: Print the most similar pair and least similar pair with their cosine scores.
The most similar pair should be two complaints that describe card fraud in different words.

### Steps
1. Use a list comprehension calling `get_embedding(c)` on each complaint in `complaints_dataset`
2. Use `np.stack(all_embeddings)` to build a matrix of shape (10, 768)
3. Call `cosine_similarity(embeddings_matrix)` to get a (10, 10) similarity matrix
4. Use `np.fill_diagonal(sim_matrix, 0.0)` to zero out self-similarity on the diagonal
5. Use `np.argmax(sim_matrix)` on the flattened matrix, then `np.unravel_index(..., sim_matrix.shape)` to get the (row, col) index of the most similar pair
6. Use `np.argmin(sim_matrix)` and `np.unravel_index` the same way for the least similar pair
7. Print both pairs with their cosine scores

In [ ]:
# Lab 2 -- Complaint Similarity Matrix (SOLUTION)
# Use get_embedding() and cosine_similarity() from above.

complaints_dataset = [
    "I was charged twice for the same transaction.",
    "A duplicate payment was debited from my current account.",
    "Someone used my card without my permission.",
    "Fraudulent transactions appeared on my statement last week.",
    "My mortgage application has been stuck for six weeks.",
    "No one has updated me on my home loan status.",
    "The Barclays app keeps crashing when I try to log in.",
    "Your mobile app is completely broken on my Android phone.",
    "I need to increase my overdraft limit but no one is responding.",
    "I've been trying to raise my overdraft for two months.",
]

# Step 1: Compute embeddings for all 10 complaints using get_embedding()
# List comprehension: applies get_embedding to each complaint, returns list of (768,) arrays
all_embeddings = [get_embedding(c) for c in complaints_dataset]

# Step 2: Stack into a numpy matrix of shape (10, 768)
# np.stack creates a new axis 0, turning a list of (768,) arrays into (10, 768)
embeddings_matrix = np.stack(all_embeddings)

# Step 3: Compute the full pairwise cosine similarity matrix of shape (10, 10)
# Each entry [i, j] = cosine_similarity(complaints[i], complaints[j])
sim_matrix = cosine_similarity(embeddings_matrix)

# Step 4: Zero out the diagonal so self-similarity (1.0) does not dominate argmax
# np.fill_diagonal modifies the array IN PLACE; no assignment needed
np.fill_diagonal(sim_matrix, 0.0)

# Step 5: Find the most similar pair
# np.argmax on flattened matrix gives the linear index; np.unravel_index converts to (row, col)
flat_max = np.argmax(sim_matrix)
most_similar_idx = np.unravel_index(flat_max, sim_matrix.shape)

# Step 6: Find the least similar pair
# np.argmin on the same zero-diagonal matrix gives the least similar pair
flat_min = np.argmin(sim_matrix)
least_similar_idx = np.unravel_index(flat_min, sim_matrix.shape)

# Step 7: Print results
i, j = most_similar_idx
print("Most similar pair:")
print(f"  [{i}] {complaints_dataset[i]}")
print(f"  [{j}] {complaints_dataset[j]}")
print(f"  Cosine similarity: {sim_matrix[i, j]:.4f}")
print()
p, q = least_similar_idx
print("Least similar pair:")
print(f"  [{p}] {complaints_dataset[p]}")
print(f"  [{q}] {complaints_dataset[q]}")
print(f"  Cosine similarity: {sim_matrix[p, q]:.4f}")

In [ ]:
# Lab 2 verification
assert all_embeddings is not None, "all_embeddings is still None"
assert embeddings_matrix is not None, "embeddings_matrix is still None"
assert sim_matrix is not None, "sim_matrix is still None"

assert len(all_embeddings) == 10, f"Expected 10 embeddings, got {len(all_embeddings)}"
assert embeddings_matrix.shape == (10, 768), f"Expected (10, 768), got {embeddings_matrix.shape}"
assert sim_matrix.shape == (10, 10), f"Expected (10, 10), got {sim_matrix.shape}"

# Diagonal should be 0.0 (zeroed out) not 1.0
assert abs(sim_matrix[0, 0]) < 0.01, "Diagonal was not zeroed out"

# Most similar pair should be within the same complaint category
i, j = most_similar_idx
max_score = sim_matrix[i, j]
assert max_score > 0.80, f"Most similar pair score {max_score:.3f} is suspiciously low. Check embeddings."

print("Lab 2 passed!")
print(f"Most similar pair score: {max_score:.4f}")
print(f"Complaint [{i}] and complaint [{j}] are semantically closest.")

In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
if all_embeddings is None or sim_matrix is None or most_similar_idx is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")

    all_embeddings = [get_embedding(c) for c in complaints_dataset]
    embeddings_matrix = np.stack(all_embeddings)
    sim_matrix = cosine_similarity(embeddings_matrix)
    np.fill_diagonal(sim_matrix, 0.0)

    flat_idx = np.argmax(sim_matrix)
    most_similar_idx = np.unravel_index(flat_idx, sim_matrix.shape)

    np.fill_diagonal(sim_matrix, 1.0)  # restore diagonal for least-similar search
    flat_idx_min = np.argmin(sim_matrix)
    least_similar_idx = np.unravel_index(flat_idx_min, sim_matrix.shape)
    np.fill_diagonal(sim_matrix, 0.0)

    i, j = most_similar_idx
    print(f"Most similar: [{i}] and [{j}], score={sim_matrix[i,j]:.4f}")
    print("Safety-net complete.")

### Lab 2 Stretch (for fast finishers)

Replace `DistilBertModel` with a sentence-transformers model:

```python
# pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("all-MiniLM-L6-v2")
st_embeddings = st_model.encode(complaints_dataset)
```

Compare the cosine similarity matrix from sentence-transformers vs DistilBERT mean pooling.
Which one produces higher similarity scores for paraphrase pairs?
Why are sentence-transformers models better for semantic similarity? (Hint: they are
trained with a contrastive loss designed for similarity, not masked language modeling.)

### Homework Extension

Build a complaint lookup function:
Given a new complaint text and a dataset of 50 historical complaints (you can repeat
the 10 above 5 times with slight variations), return the top 3 most similar complaints
with their cosine scores. This is the foundation of a retrieval-augmented support system:
find similar resolved complaints to generate a suggested resolution for the new one.

Deliverable: a function `find_similar_complaints(query, dataset, top_k=3) -> list[dict]`
where each dict has keys: text, score, rank.

## Peer Discussion (3 min)

1. You just built the core of a "semantic search" system using embeddings.
   At Barclays scale (100,000 complaints per month), what are the computational
   bottlenecks? Where would you add caching? (Think: do embeddings change if the
   model does not change?)

2. DistilBERT is 66M parameters. GPT-3 is 175B parameters.
   Why might a SMALLER model sometimes be BETTER for production embedding search?
   Think about latency, cost, and batch throughput.

3. We used mean pooling to collapse a [1, seq_len, 768] tensor to [768].
   What information might we lose? What if two complaints are identical except
   one mentions "urgent" at the start -- will the embedding capture that?

## Section 3 -- The Three Transformer Families

Now we understand tokenization and embeddings.
The deeper question: what happens inside the model between tokens-in and embeddings-out?

All transformers share the same tokenization front-end and the same self-attention
building block. They differ in HOW they wire those blocks together.

There are exactly three families that matter in practice:

| Family | Architecture | Trained to do | Famous models |
|--------|-------------|----------------|---------------|
| Encoder-only | Reads full sequence bidirectionally | Understand text | BERT, DistilBERT, RoBERTa |
| Decoder-only | Reads left-to-right, generates tokens | Generate text | GPT-2, GPT-4, LLaMA, Gemini |
| Encoder-decoder | Encodes input, decodes output | Transform text | T5, Flan-T5, BART |

For the Barclays complaint system:
- Classifying complaint category -> encoder-only (BERT-style)
- Generating a draft response -> decoder-only (GPT-style)
- Summarizing a long complaint -> encoder-decoder (T5-style)

### Beat 1 -- Using the Wrong Architecture for the Task (Broken)

What happens if we try to use a GPT-style (decoder-only) model for classification?

In [ ]:
# Beat 1 -- Using distilgpt2 (decoder-only) for complaint classification.
# GPT models are trained to PREDICT THE NEXT TOKEN, not classify sequences.
# Watch what happens when we try to use it the wrong way.

print("Loading distilgpt2 (decoder-only GPT-style model)...")
gpt_tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token  # GPT2 has no pad token by default

# The "wrong" way: feed a complaint and ask for a classification label
complaint_to_classify = "I was charged twice for the same transaction. Please refund immediately."

# GPT will just continue the text -- it has no concept of "output a label"
gpt_inputs = gpt_tokenizer(complaint_to_classify, return_tensors="pt")

gpt_model = GPT2LMHeadModel.from_pretrained("distilgpt2")
gpt_model.eval()

with torch.no_grad():
    output_ids = gpt_model.generate(
        gpt_inputs["input_ids"],
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=gpt_tokenizer.eos_token_id,
    )

generated_text = gpt_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\nInput complaint:")
print(f"  {complaint_to_classify}")
print("\nGPT-2 output (just continued the text):")
print(f"  {generated_text}")
print()
print("PROBLEM: GPT did not classify. It continued the sequence with more text.")
print("Decoder-only models are autoregressive text GENERATORS, not classifiers.")
print("For classification, we need an ENCODER-only model (BERT-style).")

### Beat 2 -- Why Three Families?

The failure above is not a bug in the code -- it is a mismatch between what the model was
trained to do and what we asked it to do.

Each transformer family solves a different problem because each was trained with a
different objective:

- **Encoder-only** (BERT-style): trained to fill in masked words anywhere in a sentence.
  The model reads the full sequence in both directions, so every token can "see" every
  other token. This makes encoder outputs ideal for understanding tasks: classification,
  similarity, and named-entity recognition.

- **Decoder-only** (GPT-style): trained to predict the next token given all previous
  tokens -- strictly left-to-right. The model learns a probability distribution over
  sequences, which is exactly what you need for text generation. Asking it to output a
  class label is like asking a calculator to describe a sunset.

- **Encoder-decoder** (T5/BART-style): combines both. The encoder reads and compresses
  the input; the decoder generates a new sequence conditioned on that compressed
  representation. This is the natural fit for sequence-to-sequence tasks: summarization,
  translation, and question answering over long context.

Refer back to the transformer-families diagram in Section 1 -- it maps each family to
its training objective and the tasks that objective suits. Choosing the wrong family adds
complexity without improving accuracy, and in some cases (like the GPT classifier above)
produces nonsense output.

In Beat 3 below, we use the HuggingFace `pipeline` abstraction to run all three families
on representative Barclays tasks and compare the outputs side by side.

In [ ]:
# Beat 3 -- Right model for each task using HuggingFace pipelines.
# Pipelines abstract the tokenize -> model -> postprocess steps.

# Task 1: Classification (encoder-only model)
print("=== Task 1: Complaint Classification (encoder-only) ===")
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",  # encoder-only, fine-tuned on NLI
    framework="pt",  # force PyTorch; avoids broken tf_keras on this image
)
complaint = "I was charged twice for the same transaction and need a refund."
labels = ["billing dispute", "fraud", "mortgage", "app issue", "general inquiry"]
result = classifier(complaint, candidate_labels=labels)
print(f"Complaint: {complaint}")
print("Predicted labels (highest score = best match):")
for label, score in zip(result["labels"][:3], result["scores"][:3]):
    print(f"  {label}: {score:.3f}")

print()

# Task 2: Text Generation (decoder-only model)
print("=== Task 2: Draft Response Generation (decoder-only) ===")
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=50256,  # GPT2 EOS token ID
    framework="pt",
)
prompt = "Dear Barclays customer, thank you for contacting us regarding your duplicate charge."
gen_result = generator(prompt)
print(f"Prompt: {prompt}")
print(f"Generated: {gen_result[0]['generated_text']}")

print()

# Task 3: Summarization (encoder-decoder model)
print("=== Task 3: Complaint Summarization (encoder-decoder) ===")
# Note: T5 is too large for ml.t3.medium; use sshleifer/distilbart-cnn-6-6 (distilled BART)
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-6-6",
    max_length=60,
    min_length=20,
    framework="pt",
)
long_complaint = (
    "I am writing to express my deep dissatisfaction with the handling of my mortgage "
    "application. I submitted all required documents over six weeks ago. Despite multiple "
    "phone calls and emails, I have received no update on the status. I am now at risk of "
    "losing my property purchase because the seller is withdrawing. This is causing me "
    "significant financial and emotional distress."
)
summary_result = summarizer(long_complaint)
print(f"Original ({len(long_complaint.split())} words):")
print(f"  {long_complaint}")
print(f"\nSummary ({len(summary_result[0]['summary_text'].split())} words):")
print(f"  {summary_result[0]['summary_text']}")

## Section 3 Exploration -- Architecture Decision Exercise

No separate lab for Section 3 (topic time budget).
Instead, work through the decision table below mentally and verify with the code cell.

For each Barclays use case, which transformer family would you choose?

| Use Case | Encoder-only | Decoder-only | Encoder-decoder | Your choice |
|----------|-------------|-------------|----------------|-------------|
| Route complaint to correct department | ? | ? | ? | |
| Generate a polite acknowledgment email | ? | ? | ? | |
| Summarize a 500-word complaint to 3 sentences | ? | ? | ? | |
| Find complaints similar to a new one (embedding search) | ? | ? | ? | |
| Extract named entities (account numbers, dates) | ? | ? | ? | |
| Translate a complaint from Welsh to English | ? | ? | ? | |

Discuss with your neighbor. Answers are in the next code cell.

In [ ]:
# Architecture Decision Reference -- answers for Section 3 exploration table.

architecture_decisions = {
    "Route complaint to correct department": {
        "choice": "Encoder-only (BERT-style)",
        "reason": "Classification over the whole input. BERT reads bidirectionally -- sees full context.",
    },
    "Generate a polite acknowledgment email": {
        "choice": "Decoder-only (GPT-style)",
        "reason": "Text generation: produce tokens autoregressively from a prompt.",
    },
    "Summarize a 500-word complaint to 3 sentences": {
        "choice": "Encoder-decoder (T5/BART-style)",
        "reason": "Seq2seq: encode the long input, decode a short output. Input and output are different.",
    },
    "Find complaints similar to a new one": {
        "choice": "Encoder-only (BERT-style)",
        "reason": "Produce a single embedding vector per sentence; cosine similarity search.",
    },
    "Extract named entities (account numbers, dates)": {
        "choice": "Encoder-only (BERT-style)",
        "reason": "Token classification: label each token independently. Needs full context.",
    },
    "Translate a complaint from Welsh to English": {
        "choice": "Encoder-decoder (T5/BART/Helsinki-NLP-style)",
        "reason": "Translation is the canonical seq2seq task. Different input and output languages.",
    },
}

for use_case, info in architecture_decisions.items():
    print(f"Use case: {use_case}")
    print(f"  --> {info['choice']}")
    print(f"  Why: {info['reason']}")
    print()

## Section 4 -- Famous Transformers: The Landscape in 2026

There are thousands of transformer models on HuggingFace Hub.
Here are the ones that matter for a developer in 2026 -- categorized by family.

### Encoder-Only (BERT family)
| Model | Params | Best for |
|-------|--------|---------|
| BERT-base | 110M | Classification, NER, QA (the original 2018 Google model) |
| DistilBERT | 66M | Faster BERT; 97% of accuracy at 60% of size |
| RoBERTa | 125M | More robustly trained BERT; better benchmarks |
| DeBERTa-v3 | 184M | Best encoder for classification and NLU as of 2024 |

### Decoder-Only (GPT family)
| Model | Params | Best for |
|-------|--------|---------|
| GPT-2 | 1.5B | Open-source text generation baseline |
| LLaMA 4 (Meta, 2025) | 17B-400B+ | Open-weight frontier model; MoE architecture |
| GPT-4o / GPT-5 (OpenAI) | unknown | Frontier closed-source; multimodal |
| Claude Sonnet 4 (Anthropic) | unknown | Frontier closed-source; long context |
| Gemini 3 (Google) | unknown | Frontier closed-source; multimodal |
| DeepSeek-V3 | 671B MoE | Open-weight, competitive with GPT-4 class |

### Encoder-Decoder (T5 family)
| Model | Params | Best for |
|-------|--------|---------|
| T5-base | 220M | Any text-to-text task; flexible |
| Flan-T5 | 80M-11B | Instruction-tuned T5; great for summarization and QA |
| BART | 140M | Summarization, denoising, translation |

### Key Insight for Barclays

In later fine-tuning topics, we will use **DistilBERT** for classification
and **Flan-T5** for instruction following. These are not the largest or newest models --
they are the RIGHT size for the tasks, the available infrastructure, and the budget.

The "which model" decision is a business decision as much as a technical one.

In [ ]:
# Quick sanity check: verify all three architecture families run successfully.
# All three models are already loaded from Section 3. This cell just shows
# the pattern side by side.

sample_complaint = "I have been waiting two weeks for a callback about my blocked account."

print("Three transformer families, same complaint, different tasks:\n")

# Encoder-only: classify
clf_result = classifier(sample_complaint, candidate_labels=["account issue", "fraud", "billing"])
print("1. ENCODER-ONLY (classification):")
print(f"   Top label: {clf_result['labels'][0]} ({clf_result['scores'][0]:.3f})")

# Decoder-only: generate continuation
gen_result2 = generator(
    "We are sorry to hear that " + sample_complaint,
    max_new_tokens=25,
    do_sample=False,
)
print("\n2. DECODER-ONLY (generation):")
print(f"   {gen_result2[0]['generated_text']}")

# Encoder-decoder: summarize
summary_result2 = summarizer(
    sample_complaint + " The customer has called three times. No agent picked up.",
    max_length=30,
    min_length=10,
)
print("\n3. ENCODER-DECODER (summarization):")
print(f"   {summary_result2[0]['summary_text']}")

## Section 5 -- The Generative AI Project Lifecycle

Every developer in this room has shipped a traditional ML project.
The GenAI lifecycle looks similar on the surface but has critical differences.

Let us compare them directly, using our Barclays complaint system as the example.

### Traditional ML Lifecycle (what you know)

1. **Problem definition** -- classify complaint into 8 categories
2. **Data collection** -- 50,000 labeled complaints from the archives
3. **Feature engineering** -- TF-IDF, bag of words, embeddings
4. **Train from scratch** -- logistic regression, XGBoost, or an MLP
5. **Evaluate** -- accuracy, F1, confusion matrix
6. **Deploy** -- REST API, batch job
7. **Monitor** -- watch for data drift

### GenAI Lifecycle (what you are learning)

1. **Problem definition** -- same, but also: is this a generation, classification, or retrieval task?
2. **Model selection** -- WHICH pretrained model? Encoder? Decoder? Which vendor?
3. **Adapt the model** -- prompt engineering first (free), then fine-tuning (expensive), then RAG
4. **Evaluate with new metrics** -- not just accuracy; also: hallucination rate, toxicity, latency, cost/token
5. **Deploy** -- inference endpoint, not training job
6. **Alignment and safety gate** -- responsible AI review (new step that has no equivalent in classical ML)
7. **Monitor** -- watch for model drift AND prompt injection AND output quality AND cost

### The Critical Differences

- You almost NEVER train from scratch. Model selection IS a first-class engineering step.
- Evaluation is harder: how do you measure whether a generated response is "good"?
- Cost is token-denominated: every input and output token costs money.
- Safety is not optional: LLMs can generate harmful content that a logistic regression cannot.

In [ ]:
# Generative AI lifecycle decision helper.
# For a given Barclays use case, what lifecycle path do you follow?

def genai_lifecycle_path(
    task_type,      # "classify" | "generate" | "retrieve" | "summarize" | "extract"
    data_available, # "labeled_small" | "labeled_large" | "unlabeled" | "none"
    latency_sla_ms, # required latency in milliseconds for production
):
    """
    Return the recommended GenAI lifecycle path for a Barclays use case.
    This is a simplified heuristic -- real decisions involve more factors.
    """
    path = {}

    # Step 1: Model selection
    if task_type in ("classify", "retrieve", "extract"):
        path["model_family"] = "encoder-only (DistilBERT / DeBERTa)"
    elif task_type == "generate":
        path["model_family"] = "decoder-only (GPT-style or Flan-T5)"
    elif task_type == "summarize":
        path["model_family"] = "encoder-decoder (Flan-T5 / BART)"
    else:
        path["model_family"] = "unclear -- revisit task definition"

    # Step 2: Adaptation strategy
    if data_available == "none":
        path["adaptation"] = "prompt engineering only (zero-shot)"
    elif data_available == "labeled_small":
        path["adaptation"] = "few-shot prompting or LoRA fine-tuning on small dataset"
    elif data_available == "labeled_large":
        path["adaptation"] = "full fine-tuning (fine-tuning capstone)"
    else:
        path["adaptation"] = "RAG with unlabeled data as retrieval corpus"

    # Step 3: Deployment
    if latency_sla_ms < 100:
        path["deployment"] = "small model + quantization (Topics 8/9)"
    elif latency_sla_ms < 1000:
        path["deployment"] = "DistilBERT or Flan-T5-small on ml.m5.xlarge"
    else:
        path["deployment"] = "larger model acceptable; batch inference"

    return path


# Barclays use cases
use_cases = [
    ("classify", "labeled_large", 200),
    ("generate", "none", 2000),
    ("summarize", "unlabeled", 500),
    ("retrieve", "none", 50),
]

for task, data, latency in use_cases:
    print(f"Task: {task} | Data: {data} | Latency SLA: {latency}ms")
    path = genai_lifecycle_path(task, data, latency)
    for k, v in path.items():
        print(f"  {k}: {v}")
    print()

## Peer Discussion (3 min)

1. In the GenAI lifecycle we listed "alignment and safety gate" as a mandatory step.
   What could go wrong if the Barclays complaint response generator is deployed WITHOUT
   an alignment review? Give two specific examples.

2. The lifecycle says "you almost never train from scratch."
   The optional deep-dive notebooks still show you how to build a transformer from
   scratch. Why is it useful to understand that internal machinery even if you will
   never write it yourself in production?

3. "Cost is token-denominated." You are given a budget of 10,000 GBP per month for
   LLM inference at Barclays. GPT-5 costs approximately $15 per million output tokens.
   How many complaints can you process per month if the average response is 200 output tokens?
   Is that enough? What would you do if it is not?

## Section 6 -- The Complete LLM Pipeline

We have seen three pieces in isolation:
1. **Tokenization**: text -> integers
2. **Embeddings**: integers -> dense vectors (meaning)
3. **Inference**: vectors -> output (classification label, generated text, summary)

In a real transformer, these three steps are chained:

```
Raw text
   |
[Tokenizer]  (vocabulary: ~30,000 subwords; BPE or WordPiece)
   |
Token IDs    (e.g., [101, 1045, 2001, ...])
   |
[Embedding layer]  (lookup table: each ID -> 768-dim vector)
   |
   + [Positional encoding]  (adds ORDER information)
   |
[N x Transformer blocks]  (self-attention + FFN; this is Topics 3+4)
   |
[Task head]  (encoder: CLS -> label; decoder: predict next token; enc-dec: decode sequence)
   |
Output (label probabilities, generated token, summary token)
```

The self-attention blocks in the middle are what Topics 3 and 4 are about.
They are the part where "meaning" is enriched by context.
Today we have treated the model as a black box and used it through pipelines.
Tomorrow we build the box.

In [ ]:
# Complete pipeline: text -> token IDs -> embeddings -> task output
# We will trace the shapes at each step so you can see the data flowing.

complaint_text = "My Barclaycard statement shows a charge I did not make."

print("=" * 60)
print("Complete LLM Pipeline Trace")
print("=" * 60)

# Step 1: Tokenize
print("\n[Step 1] Tokenization")
inputs = bert_tokenizer(
    complaint_text, return_tensors="pt", truncation=True, max_length=512
)
token_ids = inputs["input_ids"]
print(f"  Input text: '{complaint_text}'")
print(f"  Token IDs shape: {token_ids.shape}")     # [1, num_tokens]
print(f"  Tokens: {bert_tokenizer.convert_ids_to_tokens(token_ids[0].tolist())}")

# Step 2: Embedding lookup (first layer of the transformer)
print("\n[Step 2] Token Embedding Lookup")
with torch.no_grad():
    raw_embeddings = bert_model.embeddings.word_embeddings(token_ids)
print(f"  Embedding matrix shape: {raw_embeddings.shape}")  # [1, num_tokens, 768]
print(f"  Each token -> a 768-dimensional vector")

# Step 3: Full forward pass (all transformer blocks)
print("\n[Step 3] Full Transformer Forward Pass (all attention blocks)")
with torch.no_grad():
    outputs = bert_model(**inputs)
contextual_embeddings = outputs.last_hidden_state
print(f"  Contextual embeddings shape: {contextual_embeddings.shape}")  # [1, num_tokens, 768]
print(f"  The SAME {token_ids.shape[1]} tokens, but now enriched by context.")

# Step 4: Pool to sentence embedding
print("\n[Step 4] Mean Pooling -> Sentence Embedding")
mask = inputs["attention_mask"].unsqueeze(-1).expand(contextual_embeddings.size()).float()
sentence_embedding = (contextual_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
sentence_embedding = sentence_embedding.squeeze().numpy()
print(f"  Sentence embedding shape: {sentence_embedding.shape}")  # (768,)
print(f"  First 5 values: {sentence_embedding[:5].round(4)}")
print(f"\nPipeline complete. One sentence -> one 768-dim vector capturing its meaning.")

## Section 5.5 -- How Self-Attention Works (Concept Mini-Lesson)

We have called the middle of a transformer "N transformer blocks" and treated it as a
black box. This mini-lesson opens that box at the CONCEPT level. You will not derive
any math or write a training loop here. The goal is narrower and practical: after this
section you can read an attention-head heatmap, explain what the [CLS] token is, and
reason about why transformers handle long-range dependencies better than RNNs.

If you later want to BUILD attention and a full transformer from scratch, that is what
the optional deep-dive notebooks are for (topic_optional_attention_python,
topic_optional_attention_pytorch, topic_optional_transformers). They are not required.

### The problem attention solves

A complaint reads: "The branch told me the refund was processed, but it never arrived."
To classify this correctly the model must connect "refund" with "never arrived" - words
that are eight tokens apart. An RNN reads left to right and compresses everything seen
so far into one fixed-size hidden state, so distant words get diluted. Self-attention
removes that bottleneck: every token can look directly at every other token in one step.

### Queries, Keys, and Values

Self-attention gives each token three learned vectors:

- Query (Q): "what am I looking for?"
- Key (K): "what do I offer to others?"
- Value (V): "what information do I carry?"

For one token, the model compares its Query against every token's Key (a dot product).
High dot product means "these two are relevant to each other". Those scores are scaled
by the square root of the key dimension (this keeps gradients stable), then passed
through a softmax so they become weights that sum to 1. The token's new representation
is the weighted sum of every token's Value. In short:

  attention output = softmax( Q dot K_transposed / sqrt(d_k) ) times V

That is the whole operation. The word "fraud" ends up with a representation that has
pulled in information from "unauthorised" and "charge" because those tokens scored high
against its Query.

In [ ]:
# Concept demo: one self-attention step over a tiny complaint sequence.
# This is a CONCEPT illustration, not a from-scratch build. We use small random
# Q, K, V so you can watch the weights form. The optional deep-dive notebooks
# implement the real thing properly.
import numpy as np

np.random.seed(0)

tokens = ["the", "refund", "never", "arrived"]
d_k = 8  # key/query dimension

# Each token gets a random Query, Key, Value vector (a real model LEARNS these).
Q = np.random.randn(len(tokens), d_k)
K = np.random.randn(len(tokens), d_k)
V = np.random.randn(len(tokens), d_k)

# Step 1: score every Query against every Key, then scale by sqrt(d_k).
scores = (Q @ K.T) / np.sqrt(d_k)

# Step 2: softmax each row so the weights sum to 1.
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = weights / weights.sum(axis=1, keepdims=True)

# Step 3: each token's new vector is the weighted sum of all Values.
attended = weights @ V

print("Attention weight matrix (rows = query token, columns = key token):")
print("        " + "  ".join(f"{t:>8}" for t in tokens))
for tok, row in zip(tokens, weights):
    print(f"{tok:>8} " + "  ".join(f"{w:8.3f}" for w in row))
print()
print("Each row sums to 1.0:", np.allclose(weights.sum(axis=1), 1.0))
print("Output shape (one new vector per token):", attended.shape)
print()
print("Reading this matrix: row 'refund' shows how much 'refund' attends to each")
print("other token. After training on real data, semantically related tokens get")
print("the high weights. This same matrix is what an attention-head heatmap shows.")

### Multi-Head Attention

One set of Q/K/V vectors learns one kind of relationship. Real transformers run several
in parallel - these are the attention HEADS. DistilBERT has 12 heads per layer. One head
may learn "this pronoun refers to that noun", another "this adjective modifies that
noun". Each head produces its own weighted-sum output; the outputs are concatenated and
projected back down. When you see an attention-head visualization (you will, in Topic 4
and Topic 7), you are looking at ONE head's weight matrix - exactly the matrix the demo
above printed.

### Positional Encoding

Self-attention has no built-in sense of order: "refund never arrived" and "never refund
arrived" would look identical to it, because the weighted sum does not depend on
position. To fix this, transformers ADD a position signal to each token embedding
before the first block. This is positional encoding. The concept is all you need here:
order information is injected once, up front, so attention can use it. The exact
sinusoidal formula and the proof that it encodes relative distance are in the optional
transformers deep-dive.

### The Encoder Block

Stack these pieces and you have one encoder block:

1. Multi-head self-attention (every token looks at every token)
2. Add and LayerNorm (a residual connection plus normalization for stable training)
3. A small feed-forward network applied to each token
4. Add and LayerNorm again

DistilBERT stacks 6 of these blocks. "N transformer blocks" from Section 6 is exactly
this, repeated N times.

### The [CLS] Token

In Section 2 you saw BERT-style tokenizers prepend a special [CLS] token to every
input. Now it makes sense. [CLS] is a token with no word meaning of its own. As it
passes through the attention blocks, it attends to every real token and accumulates a
summary of the whole sequence. After the final block, the vector sitting at the [CLS]
position is used as the sentence representation: a classification head reads that single
vector to predict a label.

This is why, in Topic 4 and Topic 5, the complaint classifier takes the [CLS] vector and
feeds it to a small trainable head. The encoder does the understanding; the [CLS] vector
is where that understanding is collected; the head turns it into a Barclays complaint
category. You now have everything you need to follow that discussion.

In [ ]:
# Concept check: confirm the multi-head numbers for DistilBERT.
# This uses only the config, no model download, no training.
from transformers import DistilBertConfig

cfg = DistilBertConfig()  # the standard distilbert-base-uncased configuration
print(f"Transformer (encoder) blocks : {cfg.n_layers}")
print(f"Attention heads per block    : {cfg.n_heads}")
print(f"Hidden size                  : {cfg.dim}")
print(f"Total attention heads        : {cfg.n_layers * cfg.n_heads}")
print()
print("Each of those heads produces one attention weight matrix - one heatmap -")
print("exactly like the matrix the demo above printed. When Topic 4 visualizes an")
print("attention head, it is showing one of these.")

### Mini-Lesson Recap

- Self-attention lets every token look directly at every other token, removing the RNN
  bottleneck for long-range dependencies.
- Q, K, V: a token's Query is matched against all Keys to produce weights; the output
  is the weighted sum of Values.
- Multi-head attention runs several of these in parallel; each head is one heatmap.
- Positional encoding adds order information, because attention itself is order-blind.
- An encoder block = multi-head attention + add-and-norm + feed-forward + add-and-norm;
  DistilBERT stacks 6 of them.
- The [CLS] token accumulates a whole-sequence summary that a classification head reads.

**Peer discussion (3-5 min)**: A teammate says "we should just use an RNN, it is
simpler". Using what you just learned, give two concrete reasons a transformer encoder
will classify long Barclays complaints more reliably. Then name one cost of that choice
(hint: think about how attention scales with sequence length).

## What We Did NOT Cover Today -- And That Is Intentional

### The Self-Attention Mechanism, in Depth
We covered the core idea in the mini-lesson above: queries, keys, values, and
multi-head attention. That is enough to read attention-head visualizations and reason
about model behaviour for the rest of this course.
If you want to implement attention by hand and watch every matrix multiply, there are
two optional deep-dive notebooks (topic_optional_attention_python and
topic_optional_attention_pytorch). They are not required for the path you are on.

### Training and Fine-Tuning
Everything today was INFERENCE -- we used pretrained weights and did not update them.
In **Topic 4 (Full Fine-Tuning)** and **Topic 6 (PEFT and LoRA)** you will fine-tune
DistilBERT on a Barclays complaint dataset on a remote GPU instance.

### Positional Encoding (the Full Math)
The mini-lesson above explained why positional encoding exists and what it does. The
full sinusoidal formula and the proof that it encodes relative position are in the
optional transformers deep-dive notebook (topic_optional_transformers).

### Retrieval-Augmented Generation (RAG)
The Lab 2 Homework Extension was the first seed of RAG.
Finding similar complaints and using them as context for generation is full RAG.
This is covered in the later deployment topics.

### What You CAN Do Right Now
- Choose the right transformer family for a task
- Tokenize any text and inspect the subword vocabulary
- Get embeddings and measure semantic similarity
- Run inference with a pretrained model through HuggingFace pipelines
- Describe the GenAI project lifecycle and its unique challenges

In [ ]:
# Knowledge check -- SOLUTION answers with explanations.

answers = {
    "GPT-style (decoder-only) models are best for text classification": False,
    # False: decoder-only models generate text left-to-right; they cannot read the
    # full sequence bidirectionally. For classification, encoder-only (BERT-style)
    # models read the full context and are trained with a classification head.

    "DistilBERT has 66 million parameters, making it suitable for ml.t3.medium": True,
    # True: 66M params ~ 265MB in FP32, well within the 4GB RAM on ml.t3.medium.

    "Tokenization converts text to floating-point vectors": False,
    # False: tokenization converts text to INTEGER IDs (token IDs).
    # The EMBEDDING layer converts those integers to floating-point vectors.

    "Cosine similarity of 0.0 means two sentences are identical": False,
    # False: cosine similarity of 1.0 means identical direction (identical or proportional vectors).
    # 0.0 means the vectors are orthogonal (completely unrelated in this space).

    "The GenAI lifecycle includes a safety/alignment gate that classical ML does not": True,
    # True: LLMs can generate harmful, biased, or misleading content. Classical ML
    # models (logistic regression, XGBoost) produce scores or labels, not free text.

    "You can use the same DistilBERT model for both classification and generation": False,
    # False: DistilBERT is encoder-only. It reads the full sequence bidirectionally
    # and has no autoregressive decoder. It cannot generate new tokens.
    # For generation you need a decoder-only model (distilgpt2, LLaMA, etc.).
}

correct_answers = answers.copy()

score = 0
for statement, answer in answers.items():
    print(f"[CORRECT] {statement}")
    print(f"          Answer: {answer}")
    score += 1

print(f"\nSolution score: {score}/{len(answers)} -- all answers shown.")

## Topic 2 Wrap-Up

### Key Takeaways

1. **LLMs are transformers**. All transformers share the same tokenization front-end
   and self-attention building blocks. They differ in how those blocks are wired.

2. **Three families, three use cases**: encoder-only for understanding (BERT),
   decoder-only for generation (GPT), encoder-decoder for transformation (T5).

3. **Tokenization is not word-splitting**. Subword tokenization (BPE/WordPiece) ensures
   any text -- including Welsh financial terms -- can be represented with a fixed vocabulary.

4. **Embeddings capture meaning, not just form**. Cosine similarity on DistilBERT embeddings
   clusters semantically similar complaints, enabling the routing and search system we built.

5. **The GenAI lifecycle differs from traditional ML** in three critical ways:
   you select a pretrained model, you adapt it (don't train from scratch), and
   you have a mandatory alignment/safety gate before deployment.

### Where We Are in the Course

```
Topic 1 (done): What is GenAI?
Topic 2 (done): What is an LLM? Tokenization, embeddings, and how self-attention works.
Topic 3 (next): The HuggingFace ecosystem. Load and run pretrained models.
Topic 4:        Full fine-tuning on Barclays complaint data, and catastrophic forgetting.
Topic 5-6:      Transfer learning, then parameter-efficient fine-tuning with PEFT and LoRA.
Topic 7:        Compress and deploy the model: quantization, pruning, distillation.
Optional:       Build attention and the transformer from scratch (deep-dive notebooks).
```

In Topic 3, we move from theory to tooling: the HuggingFace ecosystem, where you load
pretrained models and run them in a few lines of code. You already have the attention
concepts you need from the mini-lesson in this topic. If you want to implement the
attention mechanism yourself in pure Python, the optional attention deep-dive notebooks
are there for you, but they are not required for what comes next.

In [ ]:
# Handoff to Topic 3: save the Barclays complaint corpus for the HuggingFace topic.
# Topic 3 loads this corpus to run pipeline() classification on real complaint text
# instead of toy strings.

# --- S3 handoff helpers ---
import json, boto3

COURSE_PREFIX = "barclays-course"

def handoff_write(bucket, topic_n, artifact, obj):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    boto3.client("s3").put_object(
        Bucket=bucket, Key=key,
        Body=json.dumps(obj, indent=2).encode("utf-8"),
    )
    print(f"Handoff written: s3://{bucket}/{key}")
    return key

complaint_corpus = {
    "complaints": test_complaints,
    "tokenizer_name": "distilbert-base-uncased",
}

handoff_write(bucket, 2, "complaint_corpus.json", complaint_corpus)
print()
print("Topic 3 will load this corpus and classify it with the HuggingFace pipeline API.")


In [ ]:
# Optional cleanup -- free memory before moving to Topic 3 (HuggingFace).
# Run this if the kernel feels slow or you are getting OOM errors.
import gc

del bert_model, gpt_model, classifier, generator, summarizer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Memory cleanup complete. Ready for Topic 3 (HuggingFace).")

## Appendix -- Model Size and Memory Reference

For ml.t3.medium (4GB RAM) and ml.g4dn.xlarge (16GB GPU VRAM):

| Model | Params | RAM (FP32) | RAM (FP16) | Fits in Studio kernel? |
|-------|--------|-----------|-----------|----------------------|
| distilgpt2 | 82M | ~330MB | ~165MB | YES |
| distilbert-base-uncased | 66M | ~265MB | ~133MB | YES |
| bert-base-uncased | 110M | ~440MB | ~220MB | YES |
| distilbart-cnn-6-6 | 306M | ~1.2GB | ~600MB | YES (marginal) |
| t5-small | 60M | ~240MB | ~120MB | YES |
| t5-base | 220M | ~880MB | ~440MB | YES |
| t5-large | 770M | ~3GB | ~1.5GB | NO (OOM on t3.medium) |
| flan-t5-base | 250M | ~1GB | ~500MB | YES |
| flan-t5-large | 780M | ~3.1GB | ~1.55GB | NO |
| llama-3-8b | 8B | ~32GB | ~16GB | YES on g4dn.xlarge FP16 |

Rule of thumb: model RAM in GB ~ params_in_billions x 4 (for FP32) or x 2 (for FP16).

In later remote training jobs, we run on ml.g4dn.xlarge (16GB VRAM) using FP16.
That is why we can use larger models for training but must use small ones for demos
in the Studio notebook kernel.

In [ ]:
# Appendix -- BPE (GPT2) vs WordPiece (BERT) tokenization side-by-side.
# Run this as the Lab 1 stretch task.

print("Loading GPT2 tokenizer (BPE)...")
gpt2_tok = GPT2Tokenizer.from_pretrained("distilgpt2")
gpt2_tok.pad_token = gpt2_tok.eos_token

financial_terms = [
    "Barclaycard",
    "overdraft",
    "unacceptable",
    "I've been charged twice",
    "fraudulently",
    "unauthorised transaction",
]

print(f"\n{'Term':<30} {'BERT (WordPiece)':<40} {'GPT2 (BPE)'}")
print("-" * 100)

for term in financial_terms:
    bert_tokens = tokenizer.tokenize(term)
    gpt2_tokens = gpt2_tok.tokenize(term)
    print(f"{term:<30} {str(bert_tokens):<40} {gpt2_tokens}")

print()
print("Observations:")
print("  - BERT uses ## prefix for continuation subwords")
print("  - GPT2 uses G (visible space) prefix for subwords that follow a space")
print("  - Both can handle any word by splitting into known subpieces")
print("  - BERT adds [CLS] and [SEP]; GPT2 adds only <|endoftext|> as delimiter")
print("  - Token counts differ between tokenizers for the same text")

In [ ]:
# Appendix -- Sentence-transformers vs DistilBERT mean pooling for similarity.
# Requires: pip install -q sentence-transformers
# Run as the Lab 2 stretch task.

try:
    from sentence_transformers import SentenceTransformer
    st_available = True
except ImportError:
    st_available = False
    print("sentence-transformers not installed. Run: pip install -q sentence-transformers")

if st_available:
    print("Loading all-MiniLM-L6-v2 (sentence-transformers)...")
    st_model = SentenceTransformer("all-MiniLM-L6-v2")

    paraphrase_pair = [
        "I was charged twice for the same transaction.",
        "A duplicate payment was debited from my account.",
    ]
    unrelated_pair = [
        "I was charged twice for the same transaction.",
        "The Barclays mobile app has a great new interface.",
    ]

    # DistilBERT mean pooling
    db_e1 = get_embedding(paraphrase_pair[0])
    db_e2 = get_embedding(paraphrase_pair[1])
    db_e3 = get_embedding(unrelated_pair[1])
    db_paraphrase_sim = cosine_similarity([db_e1], [db_e2])[0, 0]
    db_unrelated_sim  = cosine_similarity([db_e1], [db_e3])[0, 0]

    # Sentence-transformers
    st_embs = st_model.encode(paraphrase_pair + [unrelated_pair[1]])
    st_paraphrase_sim = cosine_similarity([st_embs[0]], [st_embs[1]])[0, 0]
    st_unrelated_sim  = cosine_similarity([st_embs[0]], [st_embs[2]])[0, 0]

    print("\n--- Paraphrase pair (should be HIGH) ---")
    print(f"  DistilBERT mean pooling: {db_paraphrase_sim:.4f}")
    print(f"  Sentence-transformers:   {st_paraphrase_sim:.4f}")

    print("\n--- Unrelated pair (should be LOW) ---")
    print(f"  DistilBERT mean pooling: {db_unrelated_sim:.4f}")
    print(f"  Sentence-transformers:   {st_unrelated_sim:.4f}")

    print("\nConclusion: sentence-transformers produces higher contrast (bigger gap between")
    print("paraphrase and unrelated) because it is trained with a contrastive similarity loss.")
    print("DistilBERT mean pooling works but produces less discriminative similarity scores.")

## Appendix -- GenAI vs Traditional ML Lifecycle Detailed Comparison

| Lifecycle Stage | Traditional ML | Generative AI |
|-----------------|---------------|--------------|
| Problem definition | Binary: predict X | Multi-mode: classify/generate/retrieve/summarize? |
| Data strategy | Collect + label everything | Pretrained model reduces label need; focus on evaluation data |
| Model selection | Algorithm choice (XGBoost vs NN) | Foundation model choice (size, family, vendor, license) |
| Training | Train from scratch on your data | Prompt engineering first, then fine-tuning if needed |
| Evaluation | Accuracy, F1, AUC | + BLEU, ROUGE, BERTScore, human eval, hallucination rate |
| Safety | Input validation | + output toxicity filters, PII detection, alignment review |
| Cost model | Compute cost during training | Token cost at inference (input + output, per request) |
| Deployment | Batch or real-time scoring | Chat interface, API, RAG pipeline, agent framework |
| Monitoring | Data drift, model drift | + prompt injection, output quality regression, token budget |
| Updates | Retrain on new data | Re-prompt, fine-tune delta, or swap foundation model |

### Why This Matters for Barclays

Barclays regulatory environment (FCA, PRA) adds an extra constraint: every model
decision must be auditable and explainable. The GenAI lifecycle must document:
- Which pretrained model was selected and why
- What fine-tuning data was used and whether it was reviewed for bias
- What safety filters are applied at inference time
- How human-in-the-loop review is triggered for edge cases

This is NOT a technical constraint -- it is a compliance constraint that shapes
the ENTIRE project lifecycle from day one.

In [ ]:
# Appendix -- Network connectivity test for HuggingFace Hub.
# Run this if you get "Connection error" or "Timeout" when loading models.

import urllib.request

urls = [
    "https://huggingface.co",
    "https://cdn-lfs.huggingface.co",
]

for url in urls:
    try:
        with urllib.request.urlopen(url, timeout=5) as response:
            status = response.getcode()
        print(f"[OK]   {url} returned {status}")
    except Exception as e:
        print(f"[FAIL] {url}: {e}")

print()
print("If any URL fails, ask the instructor -- Studio VPC may need an internet gateway.")
print("Public models (distilbert-base-uncased, distilgpt2) download from HuggingFace CDN.")
print("No HuggingFace token required.")

## Topic 2 Quick Reference Card

### Tokenization
```python
from transformers import DistilBertTokenizer
tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
ids = tok(text, return_tensors="pt")["input_ids"]
tokens = tok.convert_ids_to_tokens(ids[0].tolist())
decoded = tok.decode(ids[0].tolist(), skip_special_tokens=True)
```

### Embeddings
```python
from transformers import DistilBertModel
import torch, numpy as np
model = DistilBertModel.from_pretrained("distilbert-base-uncased"); model.eval()
def embed(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad(): out = model(**enc)
    mask = enc["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
    return ((out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).squeeze().numpy()
```

### Inference Pipelines
```python
from transformers import pipeline
classifier  = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")
generator   = pipeline("text-generation", model="distilgpt2")
summarizer  = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6")
```

### Architecture Decisions
| Task | Family | Example model |
|------|--------|--------------|
| Classify / embed / NER | Encoder-only | DistilBERT, DeBERTa |
| Generate / chat | Decoder-only | distilgpt2, LLaMA 4 |
| Translate / summarize | Encoder-decoder | Flan-T5, BART |

### Versions (Studio ml.t3.medium)
- transformers>=4.35.0,<4.40.0
- tokenizers>=0.15.0,<0.20.0
- numpy<2